# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [3]:
from lightningrod.training import prepare_for_training
from lightningrod.training.samples import BinaryAnswerType

default_dataset_id = "a2119549-25e6-4deb-87b2-8164949cdb61" # paste it here, or set it as an environment variable
dataset_id = config.get_config_value("DATASET_ID", default_dataset_id)

dataset = lr.datasets.get(dataset_id)
dataset.download()

[Sample(id='0016bec2-d902-42a7-8847-c1cdc4a2b59a', seed=Seed(seed_text='Title: Cursed engineering: jumping randomly through CSV files without hurting yourself\n\nContent: \n\n[Computed labels - use for answers]: Upvotes: 1Comments: 0', url=None, seed_creation_date=datetime.datetime(2026, 2, 23, 12, 50, 16, tzinfo=tzutc()), search_query="\nSELECT\n  CONCAT(\n    'Title: ', COALESCE(title, ''), '\\n\\nContent: ', COALESCE(REGEXP_REPLACE(COALESCE(text, ''), r'<[^>]*>', ''), COALESCE(url, '')),\n    '\\n\\n[Computed labels - use for answers]: ',\n    'Upvotes: ', score,\n    'Comments: ', descendants\n  ) AS content,\n  TIMESTAMP_SECONDS(time) AS time\nFROM `bigquery-public-data.hacker_news.full`\nWHERE type = 'story' AND title IS NOT NULL AND (text IS NOT NULL OR url IS NOT NULL) AND score IS NOT NULL\nORDER BY time DESC\n", additional_properties={}), question=Question(question_text='Will this post receive 5 or more comments?', question_type='QUESTION', additional_properties={}), label=La

In [4]:
import pandas as pd

train, test = prepare_for_training(
    samples=dataset.samples(),
    answer_type=BinaryAnswerType(),
    test_size=0.2,
    split_strategy="random",
)

display(pd.DataFrame(train).head())
display(pd.DataFrame(test).head())

,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,b06b2095-f926-4cf3-847c-fc578bbde2b0,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
1,580d3ed2-e3ef-46a9-9812-817da27d68b9,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
2,184b30ed-a117-46d7-be9f-6385471b854f,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
3,0f9c1523-9ca1-41b7-b303-ef8ce9869182,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
4,55e3ce25-75ab-4f31-93a6-62b6a1a6d347,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary


,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,0d0673f8-4cc7-4a19-905b-9e427842fb47,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
1,1f72f1f6-35eb-4b52-84a3-fba6dfdb5f37,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
2,0791ef9a-9950-45f7-b995-8006f2c6d2c5,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
3,23b3a88f-1718-4399-b253-6c6eb26097d3,"[{'role': 'user', 'content': 'QUESTION: Will t...",0,binary,binary_log_score,binary
4,4afdc4da-5dc1-4396-8fd1-e12d0734c263,"[{'role': 'user', 'content': 'QUESTION: Will t...",1,binary,binary_log_score,binary


In [5]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/training-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 33 rows, Test: 9 rows
Columns: ['sample_id', 'prompt', 'correct_answer', 'answer_type', 'reward_function_type', 'answer_parser_type'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/bart/training-demo/commit/cff65cd41cac66430a4ace8259dd19c6d73fdc15', commit_message='Upload dataset', commit_description='', oid='cff65cd41cac66430a4ace8259dd19c6d73fdc15', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/training-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/training-demo'), pr_revision=None, pr_num=None)

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [6]:
from lightningrod.training import TrainingConfig

DATASET_PATH = "bart/training-demo"

config = TrainingConfig(
    dataset_hf_repo=DATASET_PATH,
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=10,
)

cost_estimate = lr.training.estimate_cost(config)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.91
Effective steps: 2
Train tokens: 2,016,673
Notes: Estimate uses 0.5× max_response_length for output; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display. Use this when you want to wait for the job to finish in the notebook. Skip this if you used `create` above and prefer to poll manually.

In [ ]:
job = lr.training.run(config, name="Forecasting fine-tune", poll_interval=15)
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

## List and get jobs

List all training jobs or fetch a specific job by ID.

In [ ]:
import pandas as pd

jobs_response = lr.training.list(limit=5)

df = pd.DataFrame([
    {
        "Job ID": j.id,
        "Status": j.status,
        "Base Model": getattr(j.config, "base_model", None),
        "Trained Model ID": j.model_id,
    }
    for j in jobs_response.jobs
])

df

,Job ID,Status,Base Model,Trained Model ID
0,92e583d7-6b86-4aa2-85cf-f2802d72dc62,COMPLETED,Qwen/Qwen3-4B-Instruct-2507,checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62
1,dcfcb83a-c091-4af1-8a23-79da1e3537ce,RUNNING,Qwen/Qwen3-4B-Instruct-2507,None
2,e0813bbe-0970-4102-918e-653ad68a1026,FAILED,qwen/Qwen3-4B-Instruct-2507,None
3,b84af684-7eb0-4ebf-976e-9e0a91d01466,FAILED,qwen/Qwen3-4B-Instruct-2507,None
4,2bf4a66f-eb34-409e-b36c-f3b5d96db141,STARTING,qwen/Qwen3-4B-Instruct-2507,None


## Inference with your trained model

Once training completes, use `job.model_id` with the OpenAI-compatible API. We also have a pre-trained foresight-v3 model for forecasting — see [08_foresight_model.ipynb](08_foresight_model.ipynb).

In [ ]:
%pip install openai
from IPython.display import clear_output
clear_output()

from openai import OpenAI
from lightningrod.utils import config

base_url = config.get_config_value("LIGHTNINGROD_BASE_URL", "https://api.lightningrod.ai/api/public/v1")
client = OpenAI(api_key=api_key, base_url=f"{base_url}/openai")

In [ ]:
response = client.chat.completions.create(
    model="checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62",
    messages=[
        {"role": "system", "content": "Answer as a probability between 0 and 1 between <answer></answer> tags."},
        {"role": "user", "content": "Will the Fed cut rates by 25bp in March 2026?"}
    ]
)
print(response.choices[0].message.content)

<answer>0.15</answer>


## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset and reports metrics. Use the same dataset for a quick check, or a separate test split for production.

In [ ]:
eval_job = lr.evals.run(
    model_id="checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62",
    test_dataset_id=dataset_id,
)
print(f"Eval {eval_job.id} completed with status: {eval_job.status}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Model: checkpoint:92e583d7-6b86-4aa2-85cf-f2802d72dc62                                                       │
│    Test dataset: a2119549-25e6-4deb-87b2-8164949cdb61                                                           │
│                                                                                                                 │
│    base: {'ece': 0.781211711711727, 'n_valid': 1998, 'n_samples': 2000, 'parse_rate': 0.999, 'brier_score':     │
│  0.7231255260260259, 'mean_reward': -2.230670151721925, 'mean_valid_reward': -2.2248950467686943}               │
│    trained: {'ece': 0.07779999999999312, 'n_valid': 2000, 'n_samples': 2000, 'parse_rate': 1.0, 'brier_score':  │
│  0.11209000000000004, 'mean_reward': -0.3894061055473225, 'mean_valid_reward': -0.3894061055473225}             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Eval c9da64dd-3826-44fd-9f68-17920f4feeb9 completed with status: COMPLETED


In [ ]:
import pandas as pd

evals_response = lr.evals.list(limit=5)
pd.DataFrame([
    {
        "Eval ID": e.id,
        "Status": e.status,
        "Test Dataset": e.test_dataset_id,
        "Metrics": dict(e.metrics.additional_properties) if hasattr(e.metrics, "additional_properties") else None,
    }
    for e in evals_response.jobs
])

,Eval ID,Status,Test Dataset,Metrics
0,c9da64dd-3826-44fd-9f68-17920f4feeb9,COMPLETED,a2119549-25e6-4deb-87b2-8164949cdb61,"{'base': {'ece': 0.781211711711727, 'n_valid':..."


> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.